In [1]:
import polars as pl
from trino_connection_local_to_s3 import local_trino_engine, hive_sql, iceberg_sql

# before this make sure 
# EC2 instance is running trino connection to the schemas

In [2]:
# hive individual query
hive_sql("SHOW SCHEMAS")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-9jfbp2ina6ryyfv6ypuhpaax5q
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-9jfbp2ina6ryyfv6ypuhpaax5q.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-9jfbp2ina6ryyfv6ypuhpaax5q]


Schema
str
"""bom_nci"""
"""default"""
"""elb_logdb"""
"""information_schema"""
"""sapn2022"""
"""solar_analytics"""
"""solar_analytics_iceberg"""
"""test_db"""
"""type_probe"""


In [3]:
# iceberg indiviudal query
iceberg_sql("show schemas")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-krs6sb7i5evifhqkjjus2777le
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-krs6sb7i5evifhqkjjus2777le.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-krs6sb7i5evifhqkjjus2777le]


Schema
str
"""bom_nci"""
"""default"""
"""elb_logdb"""
"""information_schema"""
"""sapn2022"""
"""solar_analytics"""
"""solar_analytics_iceberg"""
"""system"""
"""test_db"""


In [4]:
# hive individual query
hive_sql("SHOW tables", schema = "solar_analytics")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-3x49qrhc62ughnqksndg7ng5a4
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-3x49qrhc62ughnqksndg7ng5a4.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-3x49qrhc62ughnqksndg7ng5a4]


Table
str
"""circuits"""
"""compliance_voltvar"""
"""compliance_voltwatt"""
"""meta_single_inverters"""
"""meta_single_inverters_wrong_ca…"
…
"""sites"""
"""test_sola_2025_12"""
"""test_sola_2025_7"""


In [5]:
# iceberg indiviudal query
iceberg_sql("show tables", schema = "solar_analytics_iceberg")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-srx7ar5z2gjyslpzo56eh9dciy
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-srx7ar5z2gjyslpzo56eh9dciy.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-srx7ar5z2gjyslpzo56eh9dciy]


Table
str
"""all_uncurtailedpv"""
"""all_uncurtailedpv_v2"""
"""all_uncurtailedpv_v2_flex_incl…"
"""circuits"""
"""conformance_antiisland"""
…
"""structured_data"""
"""structured_data_v2"""
"""structured_data_v2_flex_includ…"


In [6]:
# use this for continued querying of the data
catalog = "iceberg"
with local_trino_engine(catalog = catalog) as engine:
    schemas = pl.read_database(query="SHOW SCHEMAS", connection=engine)

print(schemas)


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-kgfgo7zzzfqjbsjl9i7f66v2ay
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-kgfgo7zzzfqjbsjl9i7f66v2ay.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-kgfgo7zzzfqjbsjl9i7f66v2ay]
shape: (10, 1)
┌─────────────────────────┐
│ Schema                  │
│ ---                     │
│ str                     │
╞═════════════════════════╡
│ bom_nci                 │
│ default                 │
│ elb_logdb               │
│ information_schema      │
│ sapn2022                │
│ solar_analytics         │
│ solar_analytics_iceberg │
│ system                  │
│ test_db                 │
│ type_probe              │
└─────────────────────────┘


In [7]:
# iceberg_sql("SELECT * FROM ts where circuit_id = 239893 and year=2024 and month=10 limit 10")

In [8]:
# passing schema name explicitly, same as defualt here
iceberg_sql("SELECT * FROM sites limit 10", schema = "solar_analytics_iceberg")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-pgdusu5oah7eptxgrjqf8rizqi
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-pgdusu5oah7eptxgrjqf8rizqi.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-pgdusu5oah7eptxgrjqf8rizqi]


site_id,ac_capacity_kw,postcode
i64,f64,i64
1944472430,5.0,2502
1555410224,10.0,4814
758643151,3.6,4812
1669657679,6.0,2525
1947677239,20.0,2680
565382247,10.0,4715
543357583,10.0,4740
826425816,5.0,4551
1682670886,5.0,2264


In [ ]:
site_metadata = iceberg_sql("SELECT * FROM meta_up23c limit", schema = "solar_analytics_iceberg")
print(site_metadata.shape)


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-rrv2pgqqju7o9kaz3ror9fcg2a
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-rrv2pgqqju7o9kaz3ror9fcg2a.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-rrv2pgqqju7o9kaz3ror9fcg2a]


In [ ]:
# save site metadata from s3 as in sharepoint it was not available, used meta_up23c
# but can also use hive_sql("SELECT * FROM sites limit 5", schema = "solar_analytics")
site_metadata_path = "/Users/z5137801/Documents/CICCADA/LSO_anti_islanding/conformance/datasets/Solar Analytics"
site_metadata.write_csv(site_metadata_path + "/site_metadata.csv")

(60570, 38)


In [15]:
# passing schema name explicitly, same as defualt here
hive_sql("SELECT * FROM sites limit 5", schema = "solar_analytics")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-blrsfyd8hz74divioq9p7kdhia
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-blrsfyd8hz74divioq9p7kdhia.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-blrsfyd8hz74divioq9p7kdhia]


site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
i64,str,f64,f64,f64,str,f64,f64,f64,date,f64,date,str,str,f64,bool
1944472430,"""NSW""",2502.0,150.85,-34.47,"""Endeavour""",5.18,5.0,3.0,2021-08-09,1.0,2021-08-09,"""Sungrow""","""SG5KTL""",5.0,false
1555410224,"""QLD""",4814.0,146.75,-19.305,"""Ergon""",13.28,10.0,5.0,2024-03-07,1.0,2024-03-06,"""Sungrow""","""SG10RS-ADA""",10.0,true
623277618,"""QLD""",4211.0,153.3,-27.99,"""Energex""",15.75,10.0,5.0,2024-03-18,1.0,2023-06-30,"""Fronius""","""Primo GEN24 10.0""",10.0,true
1245528685,"""QLD""",4078.0,152.95,-27.63,"""Energex""",15.0,10.0,6.8,2024-09-18,1.0,2024-09-18,"""Generic Inverter""","""10.0kW""",10.0,true
1651877625,"""NSW""",2350.0,151.7,-30.51,"""Essential""",11.55,9.6,5.0,2019-06-27,2.0,2018-11-02,"""Generic Inverter""","""5kW""",5.0,false


In [11]:
# passing schema name explicitly, same as defualt here
hive_sql("SELECT * FROM circuits limit 10", schema = "solar_analytics")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-gxuk3zanuxo6v6j93tzsvkotfe
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-gxuk3zanuxo6v6j93tzsvkotfe.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-gxuk3zanuxo6v6j93tzsvkotfe]


site_id,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv
i64,i64,i64,str,i64,str,bool
484720983,135961,84703,"""Watt Watcher""",1,"""pv_site_net""",true
484720983,135961,84704,"""Watt Watcher""",1,"""pv_site_net""",true
484720983,135961,84705,"""Watt Watcher""",1,"""pv_site_net""",true
150652475,140483,658777,"""Watt Watcher""",1,"""ac_load_net""",false
150652475,140483,658778,"""Watt Watcher""",1,"""ac_load_net""",false
150652475,140483,658782,"""Watt Watcher""",1,"""pv_site_net""",true
150652475,140483,658779,"""Watt Watcher""",1,"""ac_load_net""",false
1988982127,134898,630461,"""CATCH Power""",1,"""ac_load_net""",false
1988982127,134898,630462,"""CATCH Power""",-1,"""pv_site_net""",true


In [12]:
iceberg_sql("SELECT * FROM circuits limit 10")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-iyy2e478haijk5u8xrguv6zah4
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-iyy2e478haijk5u8xrguv6zah4.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-iyy2e478haijk5u8xrguv6zah4]


site_id,circuit_id,circuit_polarity,is_pv
i64,i64,i64,bool
484720983,84703,1,true
150652475,658777,1,false
150652475,658778,1,false
150652475,658782,1,true
150652475,658779,1,false
484720983,84704,1,true
484720983,84705,1,true
1988982127,630461,1,false
1988982127,630462,-1,true


In [13]:
# this seems like HS created this one
iceberg_sql("SELECT * FROM meta_up23c limit 10")


Starting session with SessionId: z5137801_sa@ad.unsw.edu.au-vy99biav2gfpphauvrhtpghqje
Port 18080 opened for sessionId z5137801_sa@ad.unsw.edu.au-vy99biav2gfpphauvrhtpghqje.
Waiting for connections...

Connection accepted for session [z5137801_sa@ad.unsw.edu.au-vy99biav2gfpphauvrhtpghqje]


site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_json,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv,min_time,max_time,v_95,v_05,v_99,v_01,voltage_class,m_id,avg_pf,std_pf,pf_99,pf_01,n_long,n_lat,distance_km,s_99,flex_export_detected
i64,str,i64,f64,f64,str,f64,f64,f64,datetime[μs],f64,datetime[μs],str,str,str,i64,i64,str,i64,str,bool,datetime[μs],datetime[μs],f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,bool
1233585204,"""VIC""",3140,145.35,-37.755,"""Ausnet""",5.1,5.0,null,2019-08-09 00:00:00,1.0,2019-08-08 00:00:00,"""Growatt""","""5000MTL""","""5.0""",124179,219150,"""Watt Watcher""",1,"""pv_site_net""",true,2024-01-01 00:00:00,2024-06-04 20:55:00,241.45218,232.93085,243.20941,230.46495,"""neutral-absorb""","""M17""",0.99862,0.000225,0.999304,0.9983265,145.34,-37.76,1.241018,4.614773,false
158521185,"""NSW""",2488,153.55,-28.35,"""Essential""",10.54,8.0,null,2019-10-31 00:00:00,1.0,2019-10-30 00:00:00,"""Sungrow""","""SG8K-D""","""8.0""",121143,239893,"""Watt Watcher""",1,"""pv_site_net""",true,2024-01-01 00:00:00,2025-06-06 07:10:00,251.0972,242.04462,254.72723,240.5514,"""absorb""","""M39""",0.999972,0.000029,0.999994,0.9998731,153.56,-28.36,1.569777,7.975964,false
2083958230,"""NSW""",2763,150.9,-33.75,"""Endeavour""",5.28,5.0,null,2020-01-07 00:00:00,1.0,2019-12-13 00:00:00,"""Generic Inverter""","""5kW""","""5.0""",105994,254727,"""Watt Watcher""",1,"""load_air_conditioner""",false,2024-01-01 00:00:00,2025-09-11 04:40:00,246.83057,236.5794,248.38918,232.18022,"""neutral-absorb""","""M13""",0.999996,0.000008,1.0,0.9999587,150.9,-33.74,1.11,4.930158,true
248529252,"""VIC""",3140,145.35,-37.755,"""Ausnet""",5.06,5.0,null,2020-03-04 00:00:00,1.0,2020-03-04 00:00:00,"""Generic Inverter""","""5kW""","""5.0""",127872,276769,"""Watt Watcher""",1,"""ac_load_net""",false,2024-01-01 00:00:00,2025-09-23 01:10:00,242.80365,235.09694,244.69281,233.32724,"""neutral-absorb""","""M13""",0.991457,0.006766,0.997926,0.970164,145.34,-37.76,1.241018,3.9807363,false
248529252,"""VIC""",3140,145.35,-37.755,"""Ausnet""",5.06,5.0,null,2020-03-04 00:00:00,1.0,2020-03-04 00:00:00,"""Generic Inverter""","""5kW""","""5.0""",127872,276768,"""Watt Watcher""",1,"""ac_load_net""",false,2024-01-01 00:00:00,2025-09-23 01:10:00,242.80365,235.09694,244.69281,233.32724,"""neutral-absorb""","""M13""",0.991457,0.006766,0.997926,0.970164,145.34,-37.76,1.241018,3.9807363,false
248529252,"""VIC""",3140,145.35,-37.755,"""Ausnet""",5.06,5.0,null,2020-03-04 00:00:00,1.0,2020-03-04 00:00:00,"""Generic Inverter""","""5kW""","""5.0""",127872,276765,"""Watt Watcher""",1,"""load_pool""",false,2024-01-01 00:00:00,2025-09-23 01:10:00,242.80365,235.09694,244.69281,233.32724,"""neutral-absorb""","""M13""",0.991457,0.006766,0.997926,0.970164,145.34,-37.76,1.241018,3.9807363,false
804338564,"""QLD""",4670,152.45,-24.795,"""Ergon""",5.28,5.0,null,2020-06-10 00:00:00,1.0,2020-06-09 00:00:00,"""SMA""","""Sunny Boy 5.0""","""5.0""",132056,298419,"""Watt Watcher""",1,"""load_hot_water""",false,2024-01-01 00:00:00,2025-12-02 23:35:00,243.76294,237.32028,244.42073,235.5057,"""neutral-absorb""","""M27""",0.970547,0.015425,0.993794,0.923624,152.44,-24.8,1.241018,4.819746,false
804338564,"""QLD""",4670,152.45,-24.795,"""Ergon""",5.28,5.0,null,2020-06-10 00:00:00,1.0,2020-06-09 00:00:00,"""SMA""","""Sunny Boy 5.0""","""5.0""",132056,298418,"""Watt Watcher""",1,"""pv_site_net""",true,2024-01-01 00:00:00,2025-12-02 23:35:00,243.76294,237.32028,244.42073,235.5057,"""neutral-absorb""","""M27""",0.970547,0.015425,0.993794,0.923624,152.44,-24.8,1.241018,4.819746,false
1706869440,"""NSW""",2212,151.0,-33.975,"""Ausgrid""",99.6,82.8,null,2020-07-06 00:00:00,1.0,2020-07-06 00:00:00,"""Generic Inverter""","""82.8kW""","""82.8""",121381,308033,"""Watt Watcher""",1,"""pv_site_net""",true,2024-01-01 00:00:00,2025-12-31 23:55:00,243.7343